# 01 · Quickstart & layer mapping

Register a model once, score it across domains, and record from the brain three ways: a single region (standard), the whole brain (`'all'`), and a population gathered across layers (`CompositeSelector`).

> Run on EC2 (GPU + Brain-Score data). Do **not** run on a laptop.


## Load a small VLM and score one benchmark

In [ ]:
from brainscore import load_model, load_benchmark
model = load_model('clip-vit-b-32')
print('modalities:', model.supported_modalities)
benchmark = load_benchmark('Yeatman2021-lexical_decision-image')
score = benchmark(model)
print('ROAR lexical-decision score:', float(score))

## Standard recording — one region maps to one layer

In [ ]:
print('region_layer_map:', dict(model.region_layer_map))
model.start_recording('IT')
print('recording region:', model._recording_regions,
      '-> layer(s):', model._recording_layers)

## Whole-brain recording — `start_recording('all')`
Every region in the model's `region_layer_map` is recorded in one pass; neuroids carry a `region` coord.

In [ ]:
model.start_recording('all')
print('regions:', model._recording_regions)
print('layers (deduped):', model._recording_layers)

## Composite recording — a population across layers
`CompositeSelector` gathers units from several layers into one region — the v1.5 mechanism behind functional populations that span depth.

In [ ]:
from brainscore_core import CompositeSelector
# build a composite from two real layers already in this model's map
str_layers = [v for v in model.region_layer_map.values() if isinstance(v, str)]
picked = list(dict.fromkeys(str_layers))[:2] or str_layers[:1]
sel = CompositeSelector(layers=tuple((L, None) for L in picked))
print('composite layer paths:', sel.layer_paths)
model.region_layer_map['Vc'] = sel
model.start_recording('Vc')
print('composite recording:', model._composite_recording,
      '| layers:', model._recording_layers)

## Reset state (every notebook leaves the model clean)

In [ ]:
model.reset()
print('done')